In [ ]:
# ============================================================
# CELL 2 - Install dependencies  (RUN FIRST).  *** REQUIRES INTERNET = ON ***
# ============================================================
# Recent `transformers` releases REMOVED TensorFlow classes (e.g. TFViTModel).
# We pin a TF-capable version + a compatible huggingface_hub, and install the
# `tf-keras` (Keras-2) backend so the ViT stream loads reliably. We do NOT
# reinstall tensorflow (that would risk breaking Kaggle's GPU build).
import os, subprocess, sys, urllib.request
os.environ["TF_USE_LEGACY_KERAS"] = "1"        # must be set before TF is imported

# 1) Internet must be ON: Kaggle right panel -> Settings -> Internet -> On.
#    BOTH this install AND the ViT pretrained-weight download need internet.
try:
    urllib.request.urlopen("https://pypi.org", timeout=10)
    print("Internet: OK")
except Exception as ex:
    print("!!! INTERNET APPEARS TO BE OFF:", ex)
    print("!!! Turn on: Settings -> Internet -> On, then Run > Restart & Run All.")

# 2) Install with VISIBLE errors (no silent failures).
def pip(args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("pip FAILED for", args, "\\n", (r.stderr or r.stdout)[-1500:])
    return r.returncode

pip(["tf-keras"])                                   # Keras-2 backend shim
pip(["transformers==4.44.2", "huggingface_hub==0.24.6"])   # still ships TFViTModel
for p in ["scikit-learn>=1.3.0", "matplotlib>=3.7.0", "seaborn>=0.12.0",
          "opencv-python>=4.8.0", "pandas>=2.0.0", "scipy>=1.11.0",
          "statsmodels>=0.14.0", "kaggle", "tqdm", "Pillow"]:
    pip([p])

# 3) Verify the pinned version actually landed.
try:
    import transformers
    print("transformers version:", transformers.__version__, "(need 4.44.2)")
except Exception as ex:
    print("transformers not importable yet (restart needed):", ex)

print("\\n>>> REQUIRED NEXT STEP:  Run > Restart & Run All  <<<")
print(">>> so TF re-imports under tf-keras and transformers 4.44.2 is loaded.")

In [ ]:
# ============================================================
# CELL 3 - Imports, reproducibility, configuration
# ============================================================
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")   # ensure Keras-2 backend
import re, random, json, time, shutil, warnings, platform
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm
from scipy import stats as sp_stats

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, roc_auc_score,
                             precision_recall_curve, average_precision_score)
from sklearn.manifold import TSNE
from sklearn.utils.class_weight import compute_class_weight
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ---------------- Reproducibility ----------------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---------------- Master switch ----------------
# SMOKE_TEST = True  -> tiny/fast run to verify the pipeline works end to end.
# SMOKE_TEST = False -> full experiment (multi-hour, GPU strongly recommended).
SMOKE_TEST = False

# ---------------- Dataset ----------------
# MSLD v2.0 - 6 clinically-relevant classes (differentials of Mpox rash).
CLASS_NAMES = ["Monkeypox", "Chickenpox", "Measles", "Cowpox", "HFMD", "Healthy"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE, CHANNELS = 224, 3

# ---------------- Split (patient-level, before augmentation) ----------------
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
# NOTE: augmentation is now ON-THE-FLY (per-epoch), not static duplication;
# class imbalance is handled by class weights (see Cells 9-10).

# ---------------- Training ----------------
BATCH_SIZE   = 16 if SMOKE_TEST else 32
EPOCHS       = 2  if SMOKE_TEST else 40
PATIENCE     = 1  if SMOKE_TEST else 8
K_FOLDS      = 2  if SMOKE_TEST else 5
LR_HEAD      = 5e-4
LR_MOBILENET = 1e-3
LR_VIT       = 1e-4
LR_EFFNET    = 1e-3

# ---------------- H-SAM / SAM ----------------
RHO   = 0.05    # trust-region radius
ALPHA = 0.5     # H-SAM adaptive-scaling coefficient
VIT_UNFREEZE_LAYERS = 4    # last N of 12 ViT encoder layers fine-tuned

# ---------------- Paths ----------------
DATA_ROOT   = Path("./data/msld_v2")
RESULTS_DIR = Path("./results");    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = Path("./saved_models"); MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Hardware ----------------
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

print(f"SMOKE_TEST      : {SMOKE_TEST}")
print(f"TensorFlow      : {tf.__version__}")
print(f"Python          : {platform.python_version()}")
print(f"GPU             : {[g.name for g in gpus] if gpus else 'CPU only'}")
print(f"Classes ({NUM_CLASSES}) : {CLASS_NAMES}")

In [ ]:
# ============================================================
# CELL 4 - Download MSLD v2.0 from Kaggle
# ============================================================
# Dataset: https://www.kaggle.com/datasets/joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20
# Requires ~/.kaggle/kaggle.json with a valid API token.
KAGGLE_SLUG = "joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20"

# If DATA_ROOT already holds images (e.g. a mounted Kaggle *input* dataset,
# which is READ-ONLY), do not mkdir or download - just read from it.
already = (any(DATA_ROOT.rglob("*.jpg")) or any(DATA_ROOT.rglob("*.png"))
           or any(DATA_ROOT.rglob("*.jpeg")))
if already:
    print(f"Images already present under {DATA_ROOT} - skipping download (read-only OK).")
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)   # only when writable & empty
    print("Downloading MSLD v2.0 from Kaggle ...")
    rc = os.system(f'kaggle datasets download -d {KAGGLE_SLUG} --unzip -p "{DATA_ROOT}"')
    if rc != 0:
        print("\\n[!] Kaggle download failed. Options:")
        print("    1. On Kaggle: add the dataset via 'Add Input' and set DATA_ROOT to")
        print("       /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20")
        print("    2. Locally: place kaggle.json in ~/.kaggle/ (chmod 600), or")
        print(f"       manually unzip into: {DATA_ROOT.resolve()}")

# Show the tree near the image folders so we can locate 'Original Images'
for p in sorted(DATA_ROOT.rglob("*"))[:40]:
    if p.is_dir():
        print(" ", p.relative_to(DATA_ROOT))

In [ ]:
# ============================================================
# CELL 5 - Locate ORIGINAL images, build catalog, extract patient IDs
# ============================================================
# MSLD v2.0 ships "Original Images" and "Augmented Images" folders.
# We use ONLY the original images and do our own leakage-free augmentation.
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

# Map various folder spellings to our canonical class names.
CLASS_ALIASES = {
    "monkeypox": "Monkeypox", "mpox": "Monkeypox", "mkp": "Monkeypox",
    "chickenpox": "Chickenpox", "chicken pox": "Chickenpox", "chp": "Chickenpox",
    "measles": "Measles", "msl": "Measles",
    "cowpox": "Cowpox", "cow pox": "Cowpox", "cwp": "Cowpox",
    "hfmd": "HFMD", "hand foot mouth disease": "HFMD", "handfootmouth": "HFMD",
    "healthy": "Healthy", "normal": "Healthy", "hth": "Healthy",
}

def canonical(name):
    key = name.strip().lower().replace("-", " ").replace("_", " ")
    key = re.sub(r"\s+", " ", key)
    if key in CLASS_ALIASES: return CLASS_ALIASES[key]
    return CLASS_ALIASES.get(key.replace(" ", ""), None)

def find_original_root(root):
    """Pick the subtree that contains the class folders of ORIGINAL images."""
    candidates = []
    for d in root.rglob("*"):
        if d.is_dir() and "augment" not in str(d).lower():
            classdirs = [c for c in d.iterdir() if c.is_dir() and canonical(c.name)]
            imgs = sum(1 for c in classdirs for f in c.iterdir() if f.suffix.lower() in IMG_EXT)
            if len(classdirs) >= 4 and imgs > 0:
                candidates.append((imgs, d))
    if not candidates:
        raise RuntimeError("Could not locate original class folders under " + str(root))
    candidates.sort(reverse=True)
    return candidates[0][1]

ORIG_ROOT = find_original_root(DATA_ROOT)
print("Original-images root:", ORIG_ROOT)

def extract_patient_id(cls, fname):
    """Group same-patient images so they never split across train/test.
    MSLD filenames look like 'MKP_01_01.jpg' / 'CHP01_02.png' etc.
    Heuristic: class-prefix + first integer block = patient; fallback = stem."""
    stem = Path(fname).stem
    m = re.match(r"^([A-Za-z]+)[ _-]?(\d+)", stem)
    if m:
        return f"{cls}_{int(m.group(2)):04d}"
    return f"{cls}_{stem}"          # fallback: image is its own 'patient'

catalog = defaultdict(list)          # class -> list[(path, patient_id)]
for cdir in sorted(ORIG_ROOT.iterdir()):
    if not cdir.is_dir(): continue
    cls = canonical(cdir.name)
    if cls is None: continue
    for f in sorted(cdir.iterdir()):
        if f.suffix.lower() in IMG_EXT:
            catalog[cls].append((str(f), extract_patient_id(cls, f.name)))

# Report
rows = []
for cls in CLASS_NAMES:
    items = catalog.get(cls, [])
    rows.append({"Class": cls, "Images": len(items),
                 "Unique patients (est.)": len({p for _, p in items})})
df_orig = pd.DataFrame(rows)
df_orig.loc[len(df_orig)] = ["TOTAL", df_orig["Images"].sum(),
                             df_orig["Unique patients (est.)"].sum()]
print(df_orig.to_string(index=False))
assert df_orig.iloc[:-1]["Images"].sum() > 0, "No images found - check dataset path."

In [ ]:
# ============================================================
# CELL 6 - Exploratory data analysis
# ============================================================
fig, ax = plt.subplots(figsize=(9, 4.5))
counts = [len(catalog.get(c, [])) for c in CLASS_NAMES]
bars = ax.bar(CLASS_NAMES, counts, color=sns.color_palette("Set2", NUM_CLASSES),
              edgecolor="black")
for b, c in zip(bars, counts):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, str(c),
            ha="center", va="bottom", fontweight="bold")
ax.set_title("MSLD v2.0 - Original Class Distribution (imbalanced)", fontweight="bold")
ax.set_ylabel("Number of original images")
plt.tight_layout(); plt.savefig(RESULTS_DIR/"fig_eda_distribution.png", dpi=150); plt.show()

# One sample per class
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, cls in enumerate(CLASS_NAMES):
    ax = axes[i//3][i%3]
    if catalog.get(cls):
        img = Image.open(catalog[cls][0][0]).convert("RGB").resize((224, 224))
        ax.imshow(img)
    ax.set_title(f"{cls} (n={len(catalog.get(cls, []))})", fontweight="bold")
    ax.axis("off")
plt.suptitle("MSLD v2.0 - Representative sample per class", fontweight="bold")
plt.tight_layout(); plt.savefig(RESULTS_DIR/"fig_eda_samples.png", dpi=150); plt.show()

imbalance = max(counts)/max(1, min(counts))
print(f"Class imbalance ratio (max/min): {imbalance:.1f}x  -> class weights will be used.")

In [ ]:
# ============================================================
# CELL 7 - Label index maps
# ============================================================
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print(CLASS_TO_IDX)

In [ ]:
# ============================================================
# CELL 8 - PATIENT-LEVEL split BEFORE augmentation  (core leakage fix)
# ============================================================
# GroupShuffleSplit guarantees that all images of a given patient fall
# entirely inside ONE partition. Test/val hold ONLY original images.
def flatten(catalog):
    paths, labels, groups = [], [], []
    for cls, items in catalog.items():
        for pth, pid in items:
            paths.append(pth); labels.append(CLASS_TO_IDX[cls]); groups.append(pid)
    return np.array(paths), np.array(labels), np.array(groups)

paths, labels, groups = flatten(catalog)

# 70 / 15 / 15 patient-disjoint split (two-stage GroupShuffleSplit)
gss1 = GroupShuffleSplit(n_splits=1, train_size=TRAIN_RATIO, random_state=SEED)
train_idx, temp_idx = next(gss1.split(paths, labels, groups))

rel_val = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
gss2 = GroupShuffleSplit(n_splits=1, train_size=rel_val, random_state=SEED)
val_rel, test_rel = next(gss2.split(paths[temp_idx], labels[temp_idx], groups[temp_idx]))
val_idx, test_idx = temp_idx[val_rel], temp_idx[test_rel]

def to_entries(idx):
    return [(IDX_TO_CLASS[labels[i]], paths[i], groups[i]) for i in idx]

split_train, split_val, split_test = map(to_entries, (train_idx, val_idx, test_idx))

# Sanity: no patient shared across splits
g_tr, g_va, g_te = ({groups[i] for i in idx} for idx in (train_idx, val_idx, test_idx))
assert g_tr.isdisjoint(g_va) and g_tr.isdisjoint(g_te) and g_va.isdisjoint(g_te), \
       "Patient leak detected!"
print("Patient-disjoint split verified (no patient in >1 partition).")

# ---- Table 1 ----
def per_class_counts(entries):
    c = Counter(e[0] for e in entries)
    return [c.get(cls, 0) for cls in CLASS_NAMES]

t1 = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Train (orig)": per_class_counts(split_train),
    "Val (orig)":   per_class_counts(split_val),
    "Test (orig)":  per_class_counts(split_test),
})
t1.loc[len(t1)] = ["TOTAL", *t1.iloc[:, 1:].sum().tolist()]
print("\\nTABLE 1 - Source-level split (original images, BEFORE augmentation)")
print(t1.to_string(index=False))
t1.to_csv(RESULTS_DIR/"table1_split.csv", index=False)

In [ ]:
# ============================================================
# CELL 9 - ON-THE-FLY augmentation policy (training only) + transparency
# ============================================================
# Design rationale (replaces the earlier static/offline balance-to-N scheme):
#   * On-the-fly = a DIFFERENT random view each epoch -> stronger regularisation
#     than re-showing the same saved augmented copies every epoch.
#   * Class imbalance is handled by CLASS WEIGHTS (Cell 10), NOT by duplicating
#     minority images - this avoids inflating e.g. Measles from ~38 originals to
#     600 near-duplicates (a ~16x oversample a reviewer could rightly attack).
#   * Dermatology-aware: geometric transforms + only MILD photometric jitter,
#     so diagnostically meaningful colour / erythema cues are preserved.
#   * Applied to TRAINING ONLY; val/test remain original (no leakage).

# Augmentation via KERAS PREPROCESSING LAYERS applied to training BATCHES
# (Cell 10). These layers are shape-robust - unlike per-image tf.image ops,
# they don't introduce partially-unknown shapes inside tf.data. Different
# random view each epoch. Dermatology-aware: geometry + only MILD contrast.
_AUG = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.15, fill_mode="reflect", seed=SEED),   # +/- ~54 deg
    layers.RandomZoom(0.15, fill_mode="reflect", seed=SEED),        # +/- 15%
    layers.RandomContrast(0.10, seed=SEED),                         # mild
], name="augmentation")

# ---- Table 2: composition (no static duplication; imbalance -> class weights) ----
t2 = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Train (orig)": per_class_counts(split_train),
    "Val (orig)":   per_class_counts(split_val),
    "Test (orig)":  per_class_counts(split_test),
})
t2.loc[len(t2)] = ["TOTAL", *t2.iloc[:, 1:].sum().tolist()]
print("TABLE 2 - Training uses ON-THE-FLY augmentation; val/test are ORIGINAL only.")
print("Imbalance handled by class weights (Cell 10), not by minority duplication.")
print(t2.to_string(index=False))
t2.to_csv(RESULTS_DIR/"table2_augmented.csv", index=False)

In [ ]:
# ============================================================
# CELL 10 - tf.data pipeline (raw [0,255]; models self-preprocess) + class weights
# ============================================================
# Raw [0,255] lets each backbone apply its OWN correct preprocessing internally
# (MobileNetV2, EfficientNetV2 and ViT normalise differently).
AUTOTUNE = tf.data.AUTOTUNE

# Class weights from the ORIGINAL (imbalanced) training distribution -> now they
# do the real work of counter-balancing minority classes.
_tr_labels = np.array([CLASS_TO_IDX[c] for c, _, _ in split_train])
_cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=_tr_labels)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(_cw)}
print("Class weights:", {IDX_TO_CLASS[i]: round(w, 3) for i, w in CLASS_WEIGHT.items()})

def _decode(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=CHANNELS, expand_animations=False)
    img = tf.ensure_shape(img, [None, None, CHANNELS])   # enforce rank=3
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])      # -> [224,224,3], fully static
    img = tf.cast(img, tf.float32)                        # raw [0,255]
    return img

def make_ds(entries, training=False, weighted=False, batch=BATCH_SIZE):
    """training=True -> shuffle + on-the-fly augmentation on BATCHES (Cell 9).
       Every element is (image, one_hot, sample_weight); weighted=True bakes in
       class weights (used by all models so no Keras class_weight= is needed)."""
    paths  = [e[1] for e in entries]
    labels = [CLASS_TO_IDX[e[0]] for e in entries]
    wts    = [CLASS_WEIGHT[l] if weighted else 1.0 for l in labels]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, wts))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l, w: (_decode(p), tf.one_hot(l, NUM_CLASSES), w),
                num_parallel_calls=AUTOTUNE).batch(batch)
    if training:                                          # augment batched tensors
        ds = ds.map(lambda x, y, w: (_AUG(x, training=True), y, w),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

# One weighted training view used by ALL models (class weights baked in as
# sample weights). No Keras class_weight= needed -> avoids sample/class-weight clash.
train_ds = make_ds(split_train, training=True, weighted=True)
val_ds   = make_ds(split_val)
test_ds  = make_ds(split_test)
for x, y, w in train_ds.take(1):
    print("batch:", x.shape, "range:", float(tf.reduce_min(x)), float(tf.reduce_max(x)))

In [ ]:
# ============================================================
# CELL 11 - Sharpness-aware model: STANDARD SAM vs proposed H-SAM
# ============================================================
# Both share the two-pass minimax scheme; they differ ONLY in the
# perturbation rule, so the ablation isolates H-SAM's contribution.
#
#   SAM   (Foret et al., 2021):  eps = rho * g / ||g||_2
#   ASAM  (Kwon et al., 2021):   eps = rho * T^2 g / ||T g||_2   (T = |theta|)
#   H-SAM (proposed):            eps = rho * g / ||g||_2 * ( 1 + alpha * r ),
#                                r = ||theta||_2 / (||theta||_2 + ||g||_2)
#
# H-SAM novelty:
#  (1) The adaptive factor r couples parameter- and gradient-magnitude,
#      concentrating perturbation where BOTH are large - well matched to
#      transfer learning with a frozen backbone + small trainable head,
#      whose gradient statistics differ sharply from a fully-trained net.
#  (2) Adam moments are accumulated on the PERTURBED gradient, so momentum
#      history itself is sharpness-aware (hybrid moment integration).
#  (3) Perturbation is confined to the trainable head + fine-tuned ViT
#      layers only, cutting the SAM 2x overhead to a few % in practice.

class SharpnessAwareModel(keras.Model):
    """Wraps an inner functional model with SAM / H-SAM / plain training."""
    def __init__(self, inner, mode="hsam", rho=RHO, alpha=ALPHA, **kw):
        super().__init__(**kw)
        self.inner = inner
        self.mode  = mode          # 'none' | 'sam' | 'hsam'
        self.rho   = rho
        self.alpha = alpha

    def call(self, x, training=False):
        return self.inner(x, training=training)

    def _perturb(self, grads, vars_):
        gnorm = tf.linalg.global_norm([g for g in grads if g is not None]) + 1e-12
        eps = []
        for g, v in zip(grads, vars_):
            if g is None: eps.append(None); continue
            e = self.rho * tf.cast(g, tf.float32) / gnorm
            if self.mode == "hsam":
                pn = tf.norm(tf.cast(v, tf.float32)) + 1e-12
                gl = tf.norm(tf.cast(g, tf.float32)) + 1e-12
                e = e * (1.0 + self.alpha * pn / (pn + gl))
            eps.append(e)
        return eps

    def train_step(self, data):
        x, y, sw = data
        if self.mode == "none":
            with tf.GradientTape() as tape:
                yp = self(x, training=True)
                loss = self.compiled_loss(y, yp, sample_weight=sw,
                                          regularization_losses=self.losses)
            self.optimizer.apply_gradients(
                zip(tape.gradient(loss, self.trainable_variables),
                    self.trainable_variables))
            self.compiled_metrics.update_state(y, yp, sample_weight=sw)
            return {m.name: m.result() for m in self.metrics}

        # ---- SAM / H-SAM : pass 1 at theta ----
        with tf.GradientTape() as tape:
            yp = self(x, training=True)
            loss = self.compiled_loss(y, yp, sample_weight=sw,
                                      regularization_losses=self.losses)
        tv = self.trainable_variables
        g1 = tape.gradient(loss, tv)
        eps = self._perturb(g1, tv)
        for v, e in zip(tv, eps):
            if e is not None: v.assign_add(e)
        # ---- pass 2 at theta+eps ----
        with tf.GradientTape() as tape2:
            yp2 = self(x, training=True)
            loss2 = self.compiled_loss(y, yp2, sample_weight=sw,
                                       regularization_losses=self.losses)
        g2 = tape2.gradient(loss2, tv)
        for v, e in zip(tv, eps):
            if e is not None: v.assign_sub(e)          # restore theta
        self.optimizer.apply_gradients(zip(g2, tv))    # Adam on sharpness-aware grad
        self.compiled_metrics.update_state(y, yp, sample_weight=sw)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data[0], data[1]
        sw = data[2] if len(data) > 2 else None
        yp = self(x, training=False)
        self.compiled_loss(y, yp, sample_weight=sw)
        self.compiled_metrics.update_state(y, yp, sample_weight=sw)
        return {m.name: m.result() for m in self.metrics}

print("SharpnessAwareModel defined (modes: none / sam / hsam).")

In [ ]:
# ============================================================
# CELL 12 - Backbone builders with per-stream preprocessing
# ============================================================
try:
    from transformers import TFViTModel
except ImportError as e:
    raise ImportError(
        "TFViTModel is unavailable. Recent `transformers` removed TensorFlow "
        "classes. Fix, in order: (1) turn Kaggle Internet ON (Settings>Internet); "
        "(2) run Cell 2 and confirm it prints 'transformers version: 4.44.2' with "
        "NO 'pip FAILED' lines; (3) do 'Run > Restart & Run All' so the pinned "
        "version + tf-keras backend load before TensorFlow is imported. "
        f"Original error: {e!r}")
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406])
IMAGENET_STD  = tf.constant([0.229, 0.224, 0.225])

def mobilenet_features(name="mob"):
    """Frozen MobileNetV2 -> 1280-d vector. Input: raw [0,255]."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = keras.applications.mobilenet_v2.preprocess_input(inp)   # -> [-1,1]
    bb = keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                        input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    bb.trainable = False
    x = bb(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inp, x, name=name)

def effnet_features(name="eff"):
    """Frozen EfficientNetV2-B0 -> 1280-d vector (MODERN baseline)."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = keras.applications.efficientnet_v2.preprocess_input(inp)  # expects [0,255]
    bb = keras.applications.EfficientNetV2B0(include_top=False, weights="imagenet",
                                             input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    bb.trainable = False
    x = bb(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inp, x, name=name)

def vit_features(unfreeze_last_n=VIT_UNFREEZE_LAYERS, name="vit"):
    """ViT-B/16: first (12-N) encoder layers frozen, last N fine-tuned.
       Returns 768-d CLS token. Input: raw [0,255]."""
    vit = TFViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
    vit.trainable = False
    for lyr in vit.vit.encoder.layer[12 - unfreeze_last_n:]:
        lyr.trainable = True
    vit.vit.layernorm.trainable = True

    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = layers.Rescaling(1.0/255)(inp)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    x = layers.Permute((3, 1, 2))(x)                 # (B,3,224,224)
    cls = vit(x).last_hidden_state[:, 0, :]          # CLS token
    return keras.Model(inp, cls, name=name), vit

print("Backbone builders ready (MobileNetV2 / EfficientNetV2-B0 / ViT-B16).")

In [ ]:
# ============================================================
# CELL 13 - Model factory (single + hybrid) and training utilities
# ============================================================
def build_single(kind):
    """kind in {'mobilenet','effnet','vit'} -> softmax classifier."""
    if kind == "mobilenet":
        feat = mobilenet_features(); lr = LR_MOBILENET
    elif kind == "effnet":
        feat = effnet_features();    lr = LR_EFFNET
    elif kind == "vit":
        feat, _ = vit_features();     lr = LR_VIT
    else:
        raise ValueError(kind)
    inp = feat.input
    x = layers.Dense(256, activation="relu")(feat.output)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    m = keras.Model(inp, out, name=f"{kind}_baseline")
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m

def build_hybrid(mode="hsam", return_feature_model=False):
    """Dual-stream MobileNetV2(frozen)+ViT(partial) -> concat(2048) -> MLP head.
       mode: 'none' (Adam) | 'sam' | 'hsam'."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    mob = mobilenet_features("mob_stream")(inp)      # 1280
    vit_model, _ = vit_features(name="vit_stream")
    vit = vit_model(inp)                             # 768
    fused = layers.Concatenate(name="fusion")([mob, vit])   # 2048
    h = layers.Dense(512, activation="relu", name="fc1")(fused)
    h = layers.Dropout(0.3)(h)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="out")(h)
    inner = keras.Model(inp, out, name=f"hybrid_{mode}")

    model = SharpnessAwareModel(inner, mode=mode, name=f"MpoxHSAM_{mode}")
    _ = model(tf.zeros((1, IMG_SIZE, IMG_SIZE, CHANNELS)))   # build
    model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    if return_feature_model:
        feat_model = keras.Model(inp, fused, name="fused_features")
        return model, feat_model
    return model

def n_params(m):
    tr = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    fr = int(sum(np.prod(w.shape) for w in m.non_trainable_weights))
    return tr, fr

def train(model, name, tr_ds, va_ds, epochs=EPOCHS, patience=PATIENCE, weighted=True):
    cbs = [
        callbacks.EarlyStopping("val_accuracy", patience=patience,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=3,
                                    min_lr=1e-6, verbose=1),
    ]
    print(f"\\n{'='*54}\\nTraining {name}\\n{'='*54}")
    t0 = time.time()
    # SharpnessAwareModel consumes sample weights from the dataset itself;
    # Keras `class_weight` is only needed for the plain single models.
    hist = model.fit(tr_ds, validation_data=va_ds, epochs=epochs,
                     callbacks=cbs, verbose=1)
    print(f"  wall time: {(time.time()-t0)/60:.1f} min")
    return hist

def collect_predictions(model, entries):
    """Return (y_true_int, y_prob) over an entries list (original images)."""
    ds = make_ds(entries)
    yt, yp = [], []
    for x, y, _ in ds:
        yp.append(model(x, training=False).numpy()); yt.append(y.numpy())
    yt = np.vstack(yt); yp = np.vstack(yp)
    return np.argmax(yt, 1), yp

print("Model factory + training utilities ready.")

In [ ]:
# ============================================================
# CELL 14 - Ablation: train every variant on the SAME split
# ============================================================
# A. MobileNetV2          (single, frozen backbone)
# B. EfficientNetV2-B0    (single, MODERN baseline -> answers 'old models')
# C. ViT-B/16             (single, partial fine-tune)
# D. Hybrid + Adam        (no sharpness awareness)
# E. Hybrid + SAM         (standard SAM)
# F. Hybrid + H-SAM       (PROPOSED)
variants = {
    "MobileNetV2":     lambda: build_single("mobilenet"),
    "EfficientNetV2":  lambda: build_single("effnet"),
    "ViT-B16":         lambda: build_single("vit"),
    "Hybrid+Adam":     lambda: build_hybrid("none"),
    "Hybrid+SAM":      lambda: build_hybrid("sam"),
    "Mpox-HSAM":       lambda: build_hybrid("hsam"),
}

histories, test_metrics, test_preds = {}, {}, {}
for name, builder in variants.items():
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    model = builder()
    tr, fr = n_params(model)
    print(f"\\n[{name}] trainable={tr:,}  frozen={fr:,}")
    is_single = name in ("MobileNetV2", "EfficientNetV2", "ViT-B16")
    if is_single:
        # class weights already baked into train_ds as sample weights
        hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                         callbacks=[callbacks.EarlyStopping("val_accuracy",
                                        patience=PATIENCE, restore_best_weights=True, verbose=1),
                                    callbacks.ReduceLROnPlateau("val_loss", factor=0.5,
                                        patience=3, min_lr=1e-6, verbose=0)],
                         verbose=1)
    else:
        hist = train(model, name, train_ds, val_ds)
    histories[name] = hist.history

    yt, yp = collect_predictions(model, split_test)
    acc = float(np.mean(np.argmax(yp, 1) == yt))
    test_metrics[name] = {"test_accuracy": acc}
    test_preds[name] = (yt, yp)
    try:
        test_metrics[name]["macro_auc"] = float(
            roc_auc_score(np.eye(NUM_CLASSES)[yt], yp, average="macro", multi_class="ovr"))
    except Exception:
        test_metrics[name]["macro_auc"] = float("nan")
    model.save_weights(str(MODELS_DIR/f"{name}.weights.h5"))
    print(f"  TEST accuracy={acc:.4f}  macroAUC={test_metrics[name]['macro_auc']:.4f}")

print("\\n=== Ablation (held-out test set) ===")
for n, m in test_metrics.items():
    print(f"  {n:16s} acc={m['test_accuracy']:.4f}  AUC={m['macro_auc']:.4f}")

In [ ]:
# ============================================================
# CELL 15 - Training-convergence curves
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette("tab10", len(histories))
for (name, h), col in zip(histories.items(), palette):
    if "val_accuracy" in h:
        ax[0].plot(h["val_accuracy"], "-", color=col, label=name)
        ax[0].plot(h.get("accuracy", []), "--", color=col, alpha=0.4)
    if "val_loss" in h:
        ax[1].plot(h["val_loss"], "-", color=col, label=name)
        ax[1].plot(h.get("loss", []), "--", color=col, alpha=0.4)
ax[0].set_title("Validation accuracy (solid) / train (dashed)", fontweight="bold")
ax[1].set_title("Validation loss (solid) / train (dashed)", fontweight="bold")
for a in ax: a.set_xlabel("Epoch"); a.legend(fontsize=8); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR/"fig_convergence.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# CELL 16 - 5-fold Stratified-GROUP CV for the proposed H-SAM model
# ============================================================
# Folds are patient-disjoint (StratifiedGroupKFold on patient IDs).
# Within each fold: augment the training portion only; validate on originals.
def run_group_cv(k=K_FOLDS):
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED)
    results = []
    for fold, (tr_i, va_i) in enumerate(sgkf.split(paths, labels, groups), 1):
        keras.backend.clear_session(); tf.random.set_seed(SEED+fold)
        print(f"\\n----- Fold {fold}/{k} -----")
        tr_entries = [(IDX_TO_CLASS[labels[i]], paths[i], groups[i]) for i in tr_i]
        va_entries = [(IDX_TO_CLASS[labels[i]], paths[i], groups[i]) for i in va_i]

        # On-the-fly augmentation on the fold's training portion; validate on originals.
        tr_ds = make_ds(tr_entries, training=True, weighted=True)
        va_ds = make_ds(va_entries)

        model = build_hybrid("hsam")
        model.fit(tr_ds, validation_data=va_ds, epochs=EPOCHS,
                  callbacks=[callbacks.EarlyStopping("val_accuracy", patience=PATIENCE,
                                                     restore_best_weights=True, verbose=0),
                             callbacks.ReduceLROnPlateau("val_loss", factor=0.5,
                                                         patience=3, min_lr=1e-6, verbose=0)],
                  verbose=0)
        yt, yp = collect_predictions(model, va_entries)
        acc = float(np.mean(np.argmax(yp, 1) == yt))
        try:
            auc_v = float(roc_auc_score(np.eye(NUM_CLASSES)[yt], yp,
                                        average="macro", multi_class="ovr"))
        except Exception:
            auc_v = float("nan")
        f1 = classification_report(yt, np.argmax(yp, 1), output_dict=True,
                                   zero_division=0)["macro avg"]["f1-score"]
        results.append({"fold": fold, "accuracy": acc, "macro_auc": auc_v, "macro_f1": f1})
        print(f"  acc={acc:.4f}  AUC={auc_v:.4f}  F1={f1:.4f}")
    return pd.DataFrame(results)

cv_df = run_group_cv()
print("\\n", cv_df.to_string(index=False))
cv_df.to_csv(RESULTS_DIR/"table3_cv.csv", index=False)

In [ ]:
# ============================================================
# CELL 17 - Statistics: mean +/- SD, 95% CI (t), and bootstrap CI
# ============================================================
def mean_ci(x, conf=0.95):
    x = np.asarray(x, float); n = len(x)
    m, sd = x.mean(), x.std(ddof=1) if n > 1 else 0.0
    if n > 1:
        lo, hi = sp_stats.t.interval(conf, n-1, loc=m, scale=sd/np.sqrt(n))
    else:
        lo = hi = m
    return m, sd, (lo, hi)

for col in ["accuracy", "macro_auc", "macro_f1"]:
    m, sd, (lo, hi) = mean_ci(cv_df[col])
    print(f"{col:10s}: {m:.4f} +/- {sd:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")

# Bootstrap CI on the held-out TEST set for the proposed model
yt_test, yp_test = test_preds["Mpox-HSAM"]
pred_test = np.argmax(yp_test, 1)
def bootstrap_acc(yt, pred, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed); n = len(yt); accs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        accs.append(np.mean(pred[idx] == yt[idx]))
    return np.percentile(accs, [2.5, 97.5])
blo, bhi = bootstrap_acc(yt_test, pred_test)
print(f"\\nTest accuracy (point) : {np.mean(pred_test==yt_test):.4f}")
print(f"Test accuracy bootstrap 95% CI : [{blo:.4f}, {bhi:.4f}]")

In [ ]:
# ============================================================
# CELL 18 - McNemar significance tests between models (same test set)
# ============================================================
# Tests whether the proposed model's error pattern differs significantly
# from each competitor on the SAME held-out images.
def mcnemar_test(yt, pred_a, pred_b):
    a_correct = pred_a == yt; b_correct = pred_b == yt
    n01 = int(np.sum(a_correct & ~b_correct))   # A right, B wrong
    n10 = int(np.sum(~a_correct & b_correct))   # A wrong, B right
    table = [[0, n01], [n10, 0]]
    res = mcnemar(table, exact=(n01 + n10) < 25, correction=True)
    return n01, n10, float(res.pvalue)

base = "Mpox-HSAM"
yt_b, yp_b = test_preds[base]; pred_b = np.argmax(yp_b, 1)
rows = []
for name, (yt_o, yp_o) in test_preds.items():
    if name == base: continue
    pred_o = np.argmax(yp_o, 1)
    n01, n10, p = mcnemar_test(yt_b, pred_b, pred_o)
    rows.append({"Comparison": f"{base} vs {name}",
                 f"{base}\u2713/other\u2717": n01,
                 f"{base}\u2717/other\u2713": n10,
                 "p-value": f"{p:.4g}",
                 "Significant(0.05)": "yes" if p < 0.05 else "no"})
mc_df = pd.DataFrame(rows)
print("TABLE 4 - McNemar tests (proposed vs each competitor)")
print(mc_df.to_string(index=False))
mc_df.to_csv(RESULTS_DIR/"table4_mcnemar.csv", index=False)

In [ ]:
# ============================================================
# CELL 19 - ROC and Precision-Recall curves (proposed model, test set)
# ============================================================
yt_int, yp = test_preds["Mpox-HSAM"]
Y = np.eye(NUM_CLASSES)[yt_int]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
roc_auc, pr_auc = {}, {}
colors = sns.color_palette("husl", NUM_CLASSES)
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(Y[:, i], yp[:, i]); roc_auc[cls] = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=colors[i], lw=2, label=f"{cls} ({roc_auc[cls]:.3f})")
    prec, rec, _ = precision_recall_curve(Y[:, i], yp[:, i])
    pr_auc[cls] = average_precision_score(Y[:, i], yp[:, i])
    axes[1].plot(rec, prec, color=colors[i], lw=2, label=f"{cls} ({pr_auc[cls]:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set(title="ROC per class (test)", xlabel="FPR", ylabel="TPR")
axes[1].set(title="Precision-Recall per class (test)", xlabel="Recall", ylabel="Precision")
for a in axes: a.legend(fontsize=8, loc="lower left"); a.grid(alpha=0.3)
plt.suptitle("Mpox-HSAM - ROC & PR", fontweight="bold")
plt.tight_layout(); plt.savefig(RESULTS_DIR/"fig_roc_pr.png", dpi=150); plt.show()
print(f"Macro ROC-AUC={np.mean(list(roc_auc.values())):.4f}  "
      f"Macro PR-AUC={np.mean(list(pr_auc.values())):.4f}")

In [ ]:
# ============================================================
# CELL 20 - Confusion matrix (counts + row-normalised)
# ============================================================
pred = np.argmax(yp, 1)
cm = confusion_matrix(yt_int, pred, labels=range(NUM_CLASSES))
cm_pct = cm / cm.sum(1, keepdims=True).clip(min=1) * 100
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set(title="Confusion matrix (counts)", xlabel="Predicted", ylabel="True")
sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100, ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set(title="Confusion matrix (row %)", xlabel="Predicted", ylabel="True")
for a in axes: a.tick_params(axis="x", rotation=35)
plt.suptitle("Mpox-HSAM - Confusion matrix (test)", fontweight="bold")
plt.tight_layout(); plt.savefig(RESULTS_DIR/"fig_confusion.png", dpi=150); plt.show()
print(classification_report(yt_int, pred, target_names=CLASS_NAMES, digits=4, zero_division=0))

In [ ]:
# ============================================================
# CELL 21 - Calibration: reliability diagram + Expected Calibration Error
# ============================================================
def expected_calibration_error(y_true, y_prob, n_bins=10):
    conf = y_prob.max(1); pred = y_prob.argmax(1); correct = (pred == y_true)
    bins = np.linspace(0, 1, n_bins+1); ece = 0.0
    xs, accs, confs = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0: continue
        a, c = correct[m].mean(), conf[m].mean()
        ece += (m.sum()/len(conf)) * abs(a - c)
        xs.append((lo+hi)/2); accs.append(a); confs.append(c)
    return ece, np.array(xs), np.array(accs), np.array(confs)

ece, xs, accs, confs = expected_calibration_error(yt_int, yp)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(confs, accs, "o-", color="#E74C3C", label=f"Mpox-HSAM (ECE={ece:.3f})")
ax.set(title="Reliability diagram (test)", xlabel="Mean predicted confidence",
       ylabel="Empirical accuracy", xlim=(0, 1), ylim=(0, 1))
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR/"fig_calibration.png", dpi=150); plt.show()
print(f"Expected Calibration Error (10 bins): {ece:.4f}")

In [ ]:
# ============================================================
# CELL 22 - t-SNE of fused 2048-d features (qualitative separability)
# ============================================================
# Rebuild the proposed model + its fused-feature extractor, load weights.
keras.backend.clear_session(); tf.random.set_seed(SEED)
proposed, feat_extractor = build_hybrid("hsam", return_feature_model=True)
proposed.load_weights(str(MODELS_DIR/"Mpox-HSAM.weights.h5"))

feats, labs = [], []
for x, y, _ in make_ds(split_test):
    feats.append(feat_extractor(x, training=False).numpy())
    labs.append(np.argmax(y.numpy(), 1))
feats = np.vstack(feats); labs = np.concatenate(labs)

perp = max(5, min(30, len(feats)//4))
emb = TSNE(n_components=2, perplexity=perp, random_state=SEED,
           init="pca").fit_transform(feats)
fig, ax = plt.subplots(figsize=(8, 7))
for i, cls in enumerate(CLASS_NAMES):
    m = labs == i
    ax.scatter(emb[m, 0], emb[m, 1], s=40, alpha=0.75,
               color=sns.color_palette("tab10", NUM_CLASSES)[i], label=cls)
ax.set(title="t-SNE of fused features (test)", xlabel="dim-1", ylabel="dim-2")
ax.legend(); plt.tight_layout()
plt.savefig(RESULTS_DIR/"fig_tsne.png", dpi=150); plt.show()
print("t-SNE done - well-separated clusters indicate discriminative fusion features.")

In [ ]:
# ============================================================
# CELL 23 - Grad-CAM qualitative analysis (per class)
# ============================================================
# Grad-CAM on the MobileNetV2 stream's last conv activation.
gc_backbone = keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                             input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
gc_head = keras.Sequential([layers.GlobalAveragePooling2D(),
                            layers.Dense(NUM_CLASSES, activation="softmax")])
# quick calibrate head on features (only to steer Grad-CAM class channel)
_ = gc_head(tf.zeros((1, 7, 7, 1280)))
LAST_CONV = "out_relu"

def gradcam(img255):
    x = keras.applications.mobilenet_v2.preprocess_input(img255[None].astype("float32"))
    grad_model = keras.Model(gc_backbone.input,
                             [gc_backbone.get_layer(LAST_CONV).output, gc_backbone.output])
    with tf.GradientTape() as tape:
        conv, _ = grad_model(x)
        score = tf.reduce_mean(conv)                 # class-agnostic saliency
    grads = tape.gradient(score, conv)[0]
    weights = tf.reduce_mean(grads, (0, 1))
    cam = tf.reduce_sum(conv[0] * weights, -1).numpy()
    cam = np.maximum(cam, 0); cam = cam / (cam.max() + 1e-8)
    return cv2.resize(cam, (IMG_SIZE, IMG_SIZE))

test_by_cls = defaultdict(list)
for cls, pth, _ in split_test: test_by_cls[cls].append(pth)

fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(11, 3.4*NUM_CLASSES))
for r, cls in enumerate(CLASS_NAMES):
    if not test_by_cls.get(cls):
        for c in range(3): axes[r][c].axis("off")
        continue
    img = np.array(Image.open(test_by_cls[cls][0]).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    cam = gradcam(img.astype("float32"))
    heat = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    over = cv2.addWeighted(img, 0.6, heat, 0.4, 0)
    axes[r][0].imshow(img);  axes[r][0].set_title(f"{cls} - original", fontweight="bold")
    axes[r][1].imshow(cam, cmap="jet"); axes[r][1].set_title("Grad-CAM")
    axes[r][2].imshow(over); axes[r][2].set_title("Overlay")
    for c in range(3): axes[r][c].axis("off")
plt.suptitle("Grad-CAM - lesion localisation per class", fontweight="bold")
plt.tight_layout(); plt.savefig(RESULTS_DIR/"fig_gradcam.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# CELL 24 - Robustness under real-world image corruptions
# ============================================================
def c_noise(img, s):  return np.clip(img + np.random.normal(0, s*255, img.shape), 0, 255)
def c_blur(img, k):   return cv2.GaussianBlur(img, (k, k), 0)
def c_bright(img, f): return np.clip(img*f, 0, 255)
def c_contrast(img, f):
    m = img.mean(); return np.clip((img-m)*f + m, 0, 255)
def c_jpeg(img, q):
    ok, enc = cv2.imencode(".jpg", cv2.cvtColor(img.astype("uint8"), cv2.COLOR_RGB2BGR),
                           [int(cv2.IMWRITE_JPEG_QUALITY), q])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return cv2.cvtColor(dec, cv2.COLOR_BGR2RGB).astype("float32")

CORRUPTIONS = {
    "Clean":              lambda im: im,
    "Gauss noise s=0.05": lambda im: c_noise(im, 0.05),
    "Gauss noise s=0.10": lambda im: c_noise(im, 0.10),
    "Blur 7x7":           lambda im: c_blur(im, 7),
    "Blur 15x15":         lambda im: c_blur(im, 15),
    "Brightness x0.4":    lambda im: c_bright(im, 0.4),
    "Contrast x0.4":      lambda im: c_contrast(im, 0.4),
    "JPEG q=10":          lambda im: c_jpeg(im, 10),
}

# Cache original test images once
_cache = [(CLASS_TO_IDX[cls],
           np.array(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)),
                    dtype="float32"))
          for cls, p, _ in split_test]

def eval_corruption(model, fn):
    correct = 0
    for lab, img in _cache:
        x = np.clip(fn(img.copy()), 0, 255)[None].astype("float32")
        correct += int(np.argmax(model(x, training=False).numpy()) == lab)
    return correct/len(_cache)

rob = {name: eval_corruption(proposed, fn) for name, fn in CORRUPTIONS.items()}
fig, ax = plt.subplots(figsize=(11, 5))
cols = ["#2ECC71"] + ["#E74C3C"]*(len(rob)-1)
bars = ax.barh(list(rob), list(rob.values()), color=cols, edgecolor="black")
for b, v in zip(bars, rob.values()):
    ax.text(v+0.005, b.get_y()+b.get_height()/2, f"{v:.3f}", va="center")
ax.axvline(rob["Clean"], ls="--", color="green", label=f"Clean={rob['Clean']:.3f}")
ax.set(xlim=(0, 1.08), title="Robustness to corruption - Mpox-HSAM", xlabel="Accuracy")
ax.legend(); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(RESULTS_DIR/"fig_robustness.png", dpi=150); plt.show()
pd.DataFrame([{"Corruption": k, "Accuracy": v} for k, v in rob.items()]
            ).to_csv(RESULTS_DIR/"table5_robustness.csv", index=False)
print("Mean corrupted accuracy:",
      round(np.mean([v for k, v in rob.items() if k != "Clean"]), 4))

In [ ]:
# ============================================================
# CELL 25 - TFLite deployment: FP32 & INT8 size / latency / accuracy
# ============================================================
FP32 = str(MODELS_DIR/"mpox_hsam_fp32.tflite")
INT8 = str(MODELS_DIR/"mpox_hsam_int8.tflite")
(MODELS_DIR/"labels.txt").write_text("\\n".join(CLASS_NAMES))

# Export the inner functional model (TFLite-friendly).
inner_infer = proposed.inner

def to_fp32(path):
    conv = tf.lite.TFLiteConverter.from_keras_model(inner_infer)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                      tf.lite.OpsSet.SELECT_TF_OPS]
    open(path, "wb").write(conv.convert()); return os.path.getsize(path)/1024

def rep_data():
    for _, p, _ in split_val[:100]:
        img = np.array(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)),
                       dtype="float32")
        yield [img[None]]

def to_int8(path):
    conv = tf.lite.TFLiteConverter.from_keras_model(inner_infer)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = rep_data
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                      tf.lite.OpsSet.SELECT_TF_OPS]
    open(path, "wb").write(conv.convert()); return os.path.getsize(path)/1024

def bench(path, warmup=5):
    itp = tf.lite.Interpreter(model_path=path); itp.allocate_tensors()
    di, do = itp.get_input_details()[0], itp.get_output_details()[0]
    lat, pred, lab = [], [], []
    for cls, p, _ in split_test:
        img = np.array(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)),
                       dtype="float32")[None]
        itp.set_tensor(di["index"], img)
        t0 = time.perf_counter(); itp.invoke(); lat.append((time.perf_counter()-t0)*1000)
        pred.append(int(np.argmax(itp.get_tensor(do["index"])[0]))); lab.append(CLASS_TO_IDX[cls])
    lat = np.array(lat[warmup:])
    return {"acc": float(np.mean(np.array(pred) == np.array(lab))),
            "mean_ms": lat.mean(), "std_ms": lat.std(),
            "p95_ms": np.percentile(lat, 95), "fps": 1000/lat.mean()}

print("Converting FP32 ..."); fp32_kb = to_fp32(FP32)
print("Converting INT8 (post-training quantisation) ..."); int8_kb = to_int8(INT8)
b32, b8 = bench(FP32), bench(INT8)

deploy = pd.DataFrame([
    {"Format": "TFLite FP32", "Size (MB)": round(fp32_kb/1024, 2),
     "Accuracy": round(b32["acc"], 4),
     "Latency ms (mean+/-sd)": f"{b32['mean_ms']:.1f}+/-{b32['std_ms']:.1f}",
     "P95 ms": round(b32["p95_ms"], 1), "FPS": round(b32["fps"], 1)},
    {"Format": "TFLite INT8", "Size (MB)": round(int8_kb/1024, 2),
     "Accuracy": round(b8["acc"], 4),
     "Latency ms (mean+/-sd)": f"{b8['mean_ms']:.1f}+/-{b8['std_ms']:.1f}",
     "P95 ms": round(b8["p95_ms"], 1), "FPS": round(b8["fps"], 1)},
])
print("\\nTABLE 6 - Deployment performance")
print(deploy.to_string(index=False))
deploy.to_csv(RESULTS_DIR/"table6_deployment.csv", index=False)
print(f"\\nSize reduction FP32->INT8: {fp32_kb/max(int8_kb,1):.1f}x   "
      f"Accuracy delta: {b8['acc']-b32['acc']:+.4f}")

In [ ]:
# ============================================================
# CELL 26 - Fair comparison with prior & modern methods
# ============================================================
acc_m, acc_sd, _ = mean_ci(cv_df["accuracy"])
comparison = pd.DataFrame([
    # study, year, method, dataset, classes, protocol, accuracy
    ["Vuran et al. (Diagnostics)", 2025, "SwinTransformer", "MSLD v2.0", 6, "hold-out", "93.71"],
    ["Vuran et al. (Diagnostics)", 2025, "ViT",             "MSLD v2.0", 6, "hold-out", "93.10"],
    ["Alghoraibi et al. (Sensors)",2025, "ITMAiNn (ViT+MobileViT+VGG16)", "custom", 6, "hold-out", "92.00"],
    ["Chakroborty (H-MpoxNet)",    2025, "Hybrid CNN",      "custom",    2, "hold-out", "91.49"],
    ["Nayak et al.",               2023, "ResNet-18",       "MSLD-like", 2, "hold-out", "99.49"],
    ["Thieme et al. (Nat. Med.)",  2023, "MPXV-CNN",        "139,198 img",2,"clinical",  "AUC .966"],
    ["Mpox-HSAM (ours)",           2026, "MobileNetV2+ViT+H-SAM", "MSLD v2.0", 6,
     "patient-level 5-fold CV + hold-out", f"{acc_m*100:.2f}+/-{acc_sd*100:.2f}"],
], columns=["Study", "Year", "Method", "Dataset", "Classes", "Protocol", "Accuracy (%)"])

print("="*100)
print("TABLE 7 - Comparison with prior methods")
print("CAVEAT: studies differ in dataset, #classes, preprocessing and validation")
print("protocol; figures are for CONTEXT, not a like-for-like benchmark. Only rows")
print("on MSLD v2.0 with a comparable protocol are directly comparable to ours.")
print("="*100)
print(comparison.to_string(index=False))
comparison.to_csv(RESULTS_DIR/"table7_comparison.csv", index=False)

In [ ]:
# ============================================================
# CELL 27 - Ablation + per-class summary tables
# ============================================================
abl = pd.DataFrame([
    {"Variant": n,
     "Optimiser": ("H-SAM" if n == "Mpox-HSAM" else
                   "SAM" if n == "Hybrid+SAM" else "Adam"),
     "Test Acc": round(m["test_accuracy"], 4),
     "Macro AUC": round(m["macro_auc"], 4)}
    for n, m in test_metrics.items()
])
print("TABLE 8 - Ablation summary (single split, held-out test)")
print(abl.to_string(index=False))
abl.to_csv(RESULTS_DIR/"table8_ablation.csv", index=False)

rep = classification_report(yt_int, pred, target_names=CLASS_NAMES,
                            output_dict=True, zero_division=0)
pc = pd.DataFrame([
    {"Class": c, "Precision": round(rep[c]["precision"], 4),
     "Recall": round(rep[c]["recall"], 4), "F1": round(rep[c]["f1-score"], 4),
     "Support": int(rep[c]["support"]), "ROC-AUC": round(roc_auc.get(c, float('nan')), 4)}
    for c in CLASS_NAMES])
pc.loc[len(pc)] = ["Macro avg", round(rep["macro avg"]["precision"], 4),
                   round(rep["macro avg"]["recall"], 4),
                   round(rep["macro avg"]["f1-score"], 4),
                   int(rep["macro avg"]["support"]),
                   round(np.mean(list(roc_auc.values())), 4)]
print("\\nTABLE 9 - Per-class performance (proposed model, test set)")
print(pc.to_string(index=False))
pc.to_csv(RESULTS_DIR/"table9_per_class.csv", index=False)

In [ ]:
# ============================================================
# CELL 28 - Final reproducibility & results report
# ============================================================
acc_m, acc_sd, acc_ci = mean_ci(cv_df["accuracy"])
auc_m, auc_sd, auc_ci = mean_ci(cv_df["macro_auc"])
print("="*62)
print(" MPOX-HSAM  -  FULL EXPERIMENT SUMMARY (MSLD v2.0)")
print("="*62)
print(f" Seed={SEED} | TF={tf.__version__} | Py={platform.python_version()} | "
      f"{'GPU' if gpus else 'CPU'}")
print(f" Split: patient-level {int(TRAIN_RATIO*100)}/{int(VAL_RATIO*100)}/"
      f"{int(TEST_RATIO*100)} BEFORE augmentation (no patient/aug leakage)")
print(f" Train(orig)={len(split_train)} + on-the-fly aug  Val={len(split_val)}  "
      f"Test={len(split_test)}  (val/test original only)")
print("-"*62)
print(f" 5-fold CV accuracy : {acc_m:.4f} +/- {acc_sd:.4f}  "
      f"95% CI [{acc_ci[0]:.4f}, {acc_ci[1]:.4f}]")
print(f" 5-fold CV macroAUC : {auc_m:.4f} +/- {auc_sd:.4f}")
print(f" Held-out test acc  : {np.mean(pred==yt_int):.4f}")
print(f" Calibration (ECE)  : {ece:.4f}")
print("-"*62)
print(" Ablation (test accuracy):")
for _, r in abl.iterrows():
    print(f"   {r['Variant']:16s} {r['Optimiser']:6s} {r['Test Acc']:.4f}")
print("-"*62)
print(f" TFLite INT8: {int8_kb/1024:.2f} MB | acc {b8['acc']:.4f} | "
      f"{b8['mean_ms']:.1f} ms | {b8['fps']:.1f} FPS")
print(" Artefacts in ./results/ :")
for f in sorted(RESULTS_DIR.iterdir()): print("   ", f.name)
print("="*62)